In [1]:
import nflreadpy as nfl
import pathlib
import polars as pl

In [2]:
#Shared variables

# seasons to retrieve data for
seasons_to_retrieve = [2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]

beginning_season = 2016
end_season = 2025

In [3]:
# get combined data

path = pathlib.Path.cwd() / f"combined_fantasy_data_{beginning_season}_{end_season}.parquet"

combined_frame = pl.read_parquet(str(path))
# print(combined_frame.columns)

for column in combined_frame.columns:
    print(f"Column: {column} | Dtype: {combined_frame[column].dtype}")

Column: player_id | Dtype: String
Column: player_display_name | Dtype: String
Column: position | Dtype: String
Column: team | Dtype: String
Column: season | Dtype: Int32
Column: completions | Dtype: Int32
Column: attempts | Dtype: Int32
Column: passing_yards | Dtype: Int32
Column: passing_tds | Dtype: Int32
Column: passing_interceptions | Dtype: Int32
Column: passing_2pt_conversions | Dtype: Int32
Column: carries | Dtype: Int32
Column: rushing_yards | Dtype: Int32
Column: rushing_tds | Dtype: Int32
Column: rushing_fumbles_lost | Dtype: Int32
Column: rushing_2pt_conversions | Dtype: Int32
Column: receptions | Dtype: Int32
Column: targets | Dtype: Int32
Column: receiving_yards | Dtype: Int32
Column: receiving_tds | Dtype: Int32
Column: receiving_fumbles_lost | Dtype: Int32
Column: receiving_2pt_conversions | Dtype: Int32
Column: fg_made | Dtype: Int32
Column: fg_missed | Dtype: Int32
Column: pat_made | Dtype: Int32
Column: fg_made_0_19 | Dtype: Int32
Column: fg_made_20_29 | Dtype: Int32


In [4]:
# analyse column data types
columns = ["passing_yards",
           "passing_tds",
           "passing_interceptions",
           "passing_2pt_conversions",
           # rushing
           "rushing_yards",
           "rushing_tds",
           "rushing_fumbles_lost",
           "rushing_2pt_conversions",
           # receiving
           "receptions",
           "receiving_yards",
           "receiving_tds",
           "receiving_2pt_conversions",
           "receiving_fumbles_lost",
           # kicking
           "pat_made",
           "fg_missed",
           "fg_made_0_19",
           "fg_made_20_29",
           "fg_made_30_39",
           "fg_made_40_49",
           "fg_made_50_59",
           "fg_made_60_",
           # defense / special teams
           "def_sacks",
           "def_interceptions",
           "fumble_recovery_opp",
           "def_tds",
           "special_teams_tds",
           "def_safeties"
           ]

for column in columns:
    print(f"Column: {column} | Dtype: {combined_frame[column].dtype}")

Column: passing_yards | Dtype: Int32
Column: passing_tds | Dtype: Int32
Column: passing_interceptions | Dtype: Int32
Column: passing_2pt_conversions | Dtype: Int32
Column: rushing_yards | Dtype: Int32
Column: rushing_tds | Dtype: Int32
Column: rushing_fumbles_lost | Dtype: Int32
Column: rushing_2pt_conversions | Dtype: Int32
Column: receptions | Dtype: Int32
Column: receiving_yards | Dtype: Int32
Column: receiving_tds | Dtype: Int32
Column: receiving_2pt_conversions | Dtype: Int32
Column: receiving_fumbles_lost | Dtype: Int32
Column: pat_made | Dtype: Int32
Column: fg_missed | Dtype: Int32
Column: fg_made_0_19 | Dtype: Int32
Column: fg_made_20_29 | Dtype: Int32
Column: fg_made_30_39 | Dtype: Int32
Column: fg_made_40_49 | Dtype: Int32
Column: fg_made_50_59 | Dtype: Int32
Column: fg_made_60_ | Dtype: Int32
Column: def_sacks | Dtype: Float64
Column: def_interceptions | Dtype: Int32
Column: fumble_recovery_opp | Dtype: Int32
Column: def_tds | Dtype: Int32
Column: special_teams_tds | Dtype:

In [5]:
# create scoring column

class_scoring_expression = (
    # passing
        pl.col("passing_yards") / 25
        + pl.col("passing_tds") * 4
        + pl.col("passing_interceptions") * -2
        + pl.col("passing_2pt_conversions") * 2
        # rushing
        + pl.col("rushing_yards") / 10
        + pl.col("rushing_tds") * 6
        + pl.col("rushing_fumbles_lost") * -2
        + pl.col("rushing_2pt_conversions") * 2
        # receiving
        + pl.col("receptions") * 1
        + pl.col("receiving_yards") / 10
        + pl.col("receiving_tds") * 6
        + pl.col("receiving_2pt_conversions") * 2
        + pl.col("receiving_fumbles_lost") * -2
        # kicking
        + pl.col("pat_made")
        + pl.col("fg_missed") * -1
        + pl.col("fg_made_0_19") * 3
        + pl.col("fg_made_20_29") * 3
        + pl.col("fg_made_30_39") * 3
        + pl.col("fg_made_40_49") * 4
        + pl.col("fg_made_50_59") * 5
        + pl.col("fg_made_60_") * 5
        # defense / special teams
        + pl.col("def_sacks")
        + pl.col("def_interceptions") * 2
        + pl.col("fumble_recovery_opp") * 2
        + pl.col("def_tds") * 6
        + pl.col("special_teams_tds") * 6
        + pl.col("def_safeties") * 2
    # TODO: use data aggregation to generate points allowed statistics NOTE: points allowed would require much data aggregation so it is left as a todo for now
).alias("class_ppr_scoring")

combined_frame.insert_column(len(combined_frame.columns), class_scoring_expression)

player_id,player_display_name,position,team,season,completions,attempts,passing_yards,passing_tds,passing_interceptions,passing_2pt_conversions,carries,rushing_yards,rushing_tds,rushing_fumbles_lost,rushing_2pt_conversions,receptions,targets,receiving_yards,receiving_tds,receiving_fumbles_lost,receiving_2pt_conversions,fg_made,fg_missed,pat_made,fg_made_0_19,fg_made_20_29,fg_made_30_39,fg_made_40_49,fg_made_50_59,fg_made_60_,fantasy_points,fantasy_points_ppr,games,def_sacks,def_interceptions,def_fumbles_forced,fumble_recovery_opp,def_tds,special_teams_tds,fumble_recovery_tds,def_safeties,class_ppr_scoring
str,str,str,str,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,f64,f64,i32,f64,i32,i32,i32,i32,i32,i32,i32,f64
"""00-0031718""","""Andrew Franks""","""K""","""MIA""",2016,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,16,3,41,0,8,6,1,1,0,0.0,0.0,0,0.0,0,0,0,0,0,0,0,89.0
"""00-0016919""","""Adam Vinatieri""","""K""","""IND""",2016,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,27,4,44,0,3,7,10,7,0,0.0,0.0,0,0.0,0,0,0,0,0,0,0,145.0
"""00-0028660""","""Dan Bailey""","""K""","""DAL""",2016,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,27,5,46,0,9,8,7,3,0,0.0,0.0,0,0.0,0,0,0,0,0,0,0,135.0
"""00-0019646""","""Sebastian Janikowski""","""K""","""LV""",2016,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,29,6,37,1,9,6,10,3,0,0.0,0.0,0,0.0,0,0,0,0,0,0,0,134.0
"""00-0004091""","""Phil Dawson""","""K""","""SF""",2016,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,18,3,33,0,4,6,7,1,0,0.0,0.0,0,0.0,0,0,0,0,0,0,0,93.0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""""","""""","""""","""SEA""",2025,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,17,47.0,18,9,7,1,4,2,0,127.0
"""""","""""","""""","""SF""",2025,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,17,20.0,6,13,10,1,0,0,0,58.0
"""""","""""","""""","""TB""",2025,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,17,37.0,13,12,10,2,0,0,1,97.0


In [6]:
# prepare output
year_begin = seasons_to_retrieve[0]
year_end = seasons_to_retrieve[-1]

path = pathlib.Path.cwd() / f"combined_data_with_scores_{year_begin}_{year_end}"

In [7]:
# output as parquet and csv
print(combined_frame.write_csv(str(path) + ".csv", include_header=True))
print(combined_frame.write_excel(str(path) + ".xlsx",include_header=True))
print(combined_frame.write_parquet(str(path) + ".parquet"))

None
None


In [8]:
for col in combined_frame.columns:
    print(f"\"{col}\",")

"player_id",
"player_display_name",
"position",
"team",
"season",
"completions",
"attempts",
"passing_yards",
"passing_tds",
"passing_interceptions",
"passing_2pt_conversions",
"carries",
"rushing_yards",
"rushing_tds",
"rushing_fumbles_lost",
"rushing_2pt_conversions",
"receptions",
"targets",
"receiving_yards",
"receiving_tds",
"receiving_fumbles_lost",
"receiving_2pt_conversions",
"fg_made",
"fg_missed",
"pat_made",
"fg_made_0_19",
"fg_made_20_29",
"fg_made_30_39",
"fg_made_40_49",
"fg_made_50_59",
"fg_made_60_",
"fantasy_points",
"fantasy_points_ppr",
"games",
"def_sacks",
"def_interceptions",
"def_fumbles_forced",
"fumble_recovery_opp",
"def_tds",
"special_teams_tds",
"fumble_recovery_tds",
"def_safeties",
"class_ppr_scoring",


In [9]:
# add prediction columns
columns_to_exclude = [
    "player_id",
    "player_display_name",
    "position",
    "team",
    "season",
    "completions",
    "attempts",
    "passing_interceptions",
    "passing_2pt_conversions",
    "carries",
    "rushing_yards",
    "rushing_tds",
    "rushing_fumbles_lost",
    "rushing_2pt_conversions",
    "receptions",
    "targets",
    "receiving_yards",
    "receiving_tds",
    "receiving_fumbles_lost",
    "receiving_2pt_conversions",
    "fg_made",
    "fg_missed",
    "pat_made",
    "fg_made_0_19",
    "fg_made_20_29",
    "fg_made_30_39",
    "fg_made_40_49",
    "fg_made_50_59",
    "fg_made_60_",
    "fantasy_points",
    "fantasy_points_ppr",
    "games",
    "def_sacks",
    "def_interceptions",
    "def_fumbles_forced",
    "fumble_recovery_opp",
    "def_tds",
    "special_teams_tds",
    "fumble_recovery_tds",
    "def_safeties",
]

for col, i in enumerate(combined_frame.columns):


SyntaxError: incomplete input (361619631.py, line 45)